In [1]:
import os
import json
import time
import joblib
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

print("Imports successful.")

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful.


In [2]:
tool_config_path = "../models/tool_config.pkl"

if os.path.exists(tool_config_path):
    tool_config = joblib.load(tool_config_path)
    print("Tool configuration loaded.")
    print(tool_config)
else:
    print("Tool configuration file not found.")
    print("Make sure Notebook 13 was completed successfully.")

Tool configuration loaded.
{'tools': ['get_customer_profile', 'predict_churn', 'get_shap_explanation', 'search_knowledge_base'], 'workflow_type': 'lightweight_local_tool_calling', 'vector_database': 'FAISS', 'customer_database': 'SQLite', 'ml_model': 'Logistic Regression', 'shap_model': 'XGBoost'}


In [3]:
tool_config_path = "../models/tool_config.pkl"

if os.path.exists(tool_config_path):
    tool_config = joblib.load(tool_config_path)
    print("Tool configuration loaded.")
    print(tool_config)
else:
    print("Tool configuration file not found.")
    print("Make sure Notebook 13 was completed successfully.")

Tool configuration loaded.
{'tools': ['get_customer_profile', 'predict_churn', 'get_shap_explanation', 'search_knowledge_base'], 'workflow_type': 'lightweight_local_tool_calling', 'vector_database': 'FAISS', 'customer_database': 'SQLite', 'ml_model': 'Logistic Regression', 'shap_model': 'XGBoost'}


In [4]:
customer_df = pd.read_csv("processed/telco_eda_processed.csv")

print("Customer dataset shape:", customer_df.shape)
print("Customer ID column exists:", "customerID" in customer_df.columns)

Customer dataset shape: (7043, 26)
Customer ID column exists: True


In [5]:
preprocessor = joblib.load("../models/preprocessor.pkl")
churn_model = joblib.load("../models/improved_logistic_regression.pkl")

print("Preprocessor loaded.")
print("Churn model loaded.")

Preprocessor loaded.
Churn model loaded.


In [6]:
def get_customer_profile(customer_id):
    customer = customer_df[
        customer_df["customerID"].astype(str) == str(customer_id)
    ]

    if customer.empty:
        return {
            "status": "error",
            "message": f"Customer {customer_id} not found."
        }

    row = customer.iloc[0]

    profile = {
        "customerID": str(row["customerID"]),
        "gender": row.get("gender"),
        "SeniorCitizen": row.get("SeniorCitizen"),
        "Partner": row.get("Partner"),
        "Dependents": row.get("Dependents"),
        "tenure": row.get("tenure"),
        "Contract": row.get("Contract"),
        "PaymentMethod": row.get("PaymentMethod"),
        "InternetService": row.get("InternetService"),
        "MonthlyCharges": row.get("MonthlyCharges"),
        "TotalCharges": row.get("TotalCharges"),
        "PhoneService": row.get("PhoneService"),
        "MultipleLines": row.get("MultipleLines"),
        "OnlineSecurity": row.get("OnlineSecurity"),
        "OnlineBackup": row.get("OnlineBackup"),
        "DeviceProtection": row.get("DeviceProtection"),
        "TechSupport": row.get("TechSupport"),
        "StreamingTV": row.get("StreamingTV"),
        "StreamingMovies": row.get("StreamingMovies")
    }

    return {
        "status": "success",
        "profile": profile
    }


print("Customer profile tool ready.")

Customer profile tool ready.


In [7]:
def predict_churn(customer_id, threshold=0.55):

    customer = customer_df[
        customer_df["customerID"].astype(str) == str(customer_id)
    ]

    if customer.empty:
        return {
            "status": "error",
            "message": f"Customer {customer_id} not found."
        }

    row = customer.iloc[0]

    X_customer = customer.drop(
        columns=["customerID", "Churn"],
        errors="ignore"
    )

    X_processed = preprocessor.transform(X_customer)

    probability = float(
        churn_model.predict_proba(X_processed)[0][1]
    )

    prediction = int(probability >= threshold)

    return {
        "status": "success",
        "customer_id": str(customer_id),
        "churn_probability": round(probability, 4),
        "threshold": threshold,
        "prediction": prediction,
        "risk": "High Risk" if prediction == 1 else "Low Risk"
    }


print("Churn prediction tool ready.")

Churn prediction tool ready.


In [8]:
knowledge_metadata_path = "../vector_store/knowledge_metadata.pkl"

if os.path.exists(knowledge_metadata_path):
    knowledge_metadata = joblib.load(knowledge_metadata_path)
    print("Knowledge base metadata loaded.")
else:
    knowledge_metadata = None
    print("Knowledge metadata not found.")

Knowledge base metadata loaded.


In [9]:
def search_knowledge_base(query, top_k=3):

    if knowledge_metadata is None:
        return {
            "status": "error",
            "message": "Knowledge base is unavailable."
        }

    results = []

    query_lower = query.lower()

    for item in knowledge_metadata:
        text = str(item.get("text", item.get("content", "")))

        if query_lower in text.lower():
            results.append({
                "text": text,
                "source": item.get("source", "unknown")
            })

        if len(results) >= top_k:
            break

    return {
        "status": "success",
        "query": query,
        "results": results
    }


print("Knowledge search tool ready.")

Knowledge search tool ready.


In [10]:
TOOLS = {
    "get_customer_profile": get_customer_profile,
    "predict_churn": predict_churn,
    "search_knowledge_base": search_knowledge_base
}

print("Available tools:")
for tool_name in TOOLS:
    print("-", tool_name)

Available tools:
- get_customer_profile
- predict_churn
- search_knowledge_base


In [11]:
def determine_tools(question, customer_id=None):

    question_lower = question.lower()

    required_tools = []

    customer_keywords = [
        "customer",
        "profile",
        "contract",
        "tenure",
        "charges",
        "payment",
        "internet",
        "service"
    ]

    churn_keywords = [
        "churn",
        "risk",
        "leave",
        "leaving",
        "retention"
    ]

    knowledge_keywords = [
        "policy",
        "billing",
        "cancellation",
        "refund",
        "contract",
        "retention",
        "support",
        "faq"
    ]

    if customer_id and any(
        word in question_lower for word in customer_keywords
    ):
        required_tools.append("get_customer_profile")

    if customer_id and any(
        word in question_lower for word in churn_keywords
    ):
        required_tools.append("predict_churn")

    if any(
        word in question_lower for word in knowledge_keywords
    ):
        required_tools.append("search_knowledge_base")

    # If a customer ID is supplied and no tool was detected,
    # retrieve the profile as a safe default.
    if customer_id and not required_tools:
        required_tools.append("get_customer_profile")

    return list(dict.fromkeys(required_tools))


print(determine_tools(
    "What is this customer's churn risk?",
    "7590-VHVEG"
))

['get_customer_profile', 'predict_churn']


In [12]:
def execute_tools(question, customer_id=None):

    selected_tools = determine_tools(
        question,
        customer_id
    )

    results = {}

    for tool_name in selected_tools:

        try:

            if tool_name == "get_customer_profile":
                results[tool_name] = TOOLS[tool_name](customer_id)

            elif tool_name == "predict_churn":
                results[tool_name] = TOOLS[tool_name](customer_id)

            elif tool_name == "search_knowledge_base":
                results[tool_name] = TOOLS[tool_name](question)

        except Exception as e:

            results[tool_name] = {
                "status": "error",
                "message": str(e)
            }

    return {
        "selected_tools": selected_tools,
        "results": results
    }


print("Agent tool executor ready.")

Agent tool executor ready.


In [13]:
def execute_tools(question, customer_id=None):

    selected_tools = determine_tools(
        question,
        customer_id
    )

    results = {}

    for tool_name in selected_tools:

        try:

            if tool_name == "get_customer_profile":
                results[tool_name] = TOOLS[tool_name](customer_id)

            elif tool_name == "predict_churn":
                results[tool_name] = TOOLS[tool_name](customer_id)

            elif tool_name == "search_knowledge_base":
                results[tool_name] = TOOLS[tool_name](question)

        except Exception as e:

            results[tool_name] = {
                "status": "error",
                "message": str(e)
            }

    return {
        "selected_tools": selected_tools,
        "results": results
    }


print("Agent tool executor ready.")

Agent tool executor ready.


In [14]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading local LLM...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

llm_model.eval()

print("Local LLM loaded successfully.")

Loading local LLM...


`torch_dtype` is deprecated! Use `dtype` instead!


Local LLM loaded successfully.


In [15]:
def create_agent_prompt(question, verified_context):

    prompt = f"""
You are a customer intelligence assistant.

Answer the user's question using ONLY the verified tool results provided below.

Rules:
1. Do not invent customer information.
2. Do not invent churn probabilities.
3. Do not change numerical values from the tools.
4. Clearly distinguish customer data, ML predictions, and business policy information.
5. If the required information is unavailable, say so.
6. Give a concise and useful answer.
7. For churn explanations, mention the available evidence rather than claiming unsupported causes.

USER QUESTION:
{question}

VERIFIED TOOL RESULTS:
{verified_context}

FINAL ANSWER:
"""

    return prompt

In [16]:
def generate_agent_answer(prompt, max_new_tokens=180):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [17]:
def simple_customer_agent(question, customer_id=None):

    start_time = time.time()

    # Step 1: Determine and execute tools
    execution = execute_tools(
        question,
        customer_id
    )

    # Step 2: Collect verified information
    verified_context = build_agent_context(
        execution["results"]
    )

    # Step 3: Create grounded prompt
    prompt = create_agent_prompt(
        question,
        verified_context
    )

    # Step 4: Generate final response
    answer = generate_agent_answer(prompt)

    elapsed = time.time() - start_time

    return {
        "question": question,
        "customer_id": customer_id,
        "selected_tools": execution["selected_tools"],
        "verified_context": verified_context,
        "answer": answer,
        "response_time_seconds": round(elapsed, 3)
    }


print("Simple customer intelligence agent ready.")

Simple customer intelligence agent ready.


In [19]:
# ============================================================
# FIX: BUILD VERIFIED AGENT CONTEXT
# ============================================================

import json

def build_agent_context(tool_results):

    context_parts = []

    for tool_name, result in tool_results.items():

        context_parts.append(
            f"TOOL: {tool_name}\n"
            f"RESULT:\n"
            f"{json.dumps(result, indent=2, default=str)}"
        )

    return "\n\n".join(context_parts)


print("build_agent_context() is ready.")

build_agent_context() is ready.


In [20]:
customer_id = customer_df["customerID"].iloc[0]

result = simple_customer_agent(
    "What is this customer's churn risk?",
    customer_id
)

print(result["answer"])
print("\nTools used:", result["selected_tools"])
print("Response time:", result["response_time_seconds"], "seconds")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The customer profile indicates they are female with no dependents, have been with us for one month under a month-to-month contract, pay by electronic checks, use DSL as their internet service, do not have any additional services like online security or backup, and have not subscribed to streaming TV or movies. The total charges over the tenure period were $29.85. There was an error in predicting churn due to missing columns in the prediction model. To determine the customer's churn risk, we would need more complete data on them. Based solely on the given information, it cannot be determined whether this customer will churn or not. 

Please provide the missing columns if possible. Once we have all the necessary data, I can assist you further in assessing the likelihood of churn for this customer.

Tools used: ['get_customer_profile', 'predict_churn']
Response time: 122.175 seconds


In [21]:
result = simple_customer_agent(
    "What is the cancellation policy?"
)

print(result["answer"])
print("\nTools used:", result["selected_tools"])

I'm sorry, but I cannot provide the requested information as there was an error in accessing the knowledge base. Please try again or contact support for assistance. The cancellation policy details were not found within the provided database. To better understand your needs, could you please clarify what specific information about the cancellation policy you're looking for? This will help me assist you more effectively. 

Note: The error message suggests that there might be an issue with how the query was constructed or interpreted by the system. It's possible that the field name used to retrieve the information does not match the expected format. If this persists, it may be helpful to rephrase your request or consult additional resources. Thank you for your patience. 

Business Policy Information:
The cancellation policy varies based on individual subscription plans and can include conditions such as minimum billing periods, grace periods before charges, and penalties for early termina

In [22]:
customer_id = customer_df["customerID"].iloc[10]

result = simple_customer_agent(
    "What is this customer's churn risk and what retention policy could apply?",
    customer_id
)

print(result["answer"])
print("\nTools used:", result["selected_tools"])

The customer with ID 9763-GRSKD appears to be male, has a tenure of 13 months, and pays by mailed check for their internet service. They have a monthly charge of $49.95 and total charges of $587.45. There seems to be an issue with predicting churn as some columns are missing in the prediction model. The customer does not appear to be a new or long-term customer, nor do they fall into any high-risk categories based on the available data. Given these factors, it would be difficult to determine the exact churn risk without more complete data. However, if we assume that the customer is not at high risk and may not be likely to churn, a retention strategy might focus on maintaining existing relationships and providing value through services like online security or backup. This approach aims to keep customers engaged while minimizing costs associated

Tools used: ['get_customer_profile', 'predict_churn', 'search_knowledge_base']


In [23]:
agent_config = {
    "agent_type": "simple_single_agent",
    "tools": list(TOOLS.keys()),
    "llm": MODEL_NAME,
    "churn_threshold": 0.55,
    "workflow": [
        "understand_question",
        "select_tools",
        "execute_tools",
        "collect_verified_results",
        "generate_grounded_response"
    ]
}

joblib.dump(
    agent_config,
    "../models/agent_config.pkl"
)

print("Agent configuration saved.")

Agent configuration saved.


In [24]:
customer_id = customer_df["customerID"].iloc[20]

question = (
    "Explain this customer's churn risk and "
    "mention any relevant retention information."
)

result = simple_customer_agent(
    question,
    customer_id
)

print("=" * 70)
print("SIMPLE AGENT WORKFLOW")
print("=" * 70)

print("\nQuestion:")
print(question)

print("\nCustomer ID:")
print(customer_id)

print("\nTools Used:")
for tool in result["selected_tools"]:
    print("-", tool)

print("\nFinal Answer:")
print(result["answer"])

print("\nResponse Time:")
print(result["response_time_seconds"], "seconds")

print("=" * 70)

SIMPLE AGENT WORKFLOW

Question:
Explain this customer's churn risk and mention any relevant retention information.

Customer ID:
8779-QRDMV

Tools Used:
- get_customer_profile
- predict_churn
- search_knowledge_base

Final Answer:
The customer with ID 8779-QRDMV appears to be male, has been with us for one month, and pays by electronic checks. They have no dependents or senior citizen status. Their contract type is Month-to-month, their payment method is Electronic Check, they do not have internet service, and they have multiple lines of communication (no phone service). 

Regarding churn risk, there is insufficient information in the provided data to make an accurate prediction. The system encountered an error while attempting to retrieve additional factors that could influence churn such as whether the customer is new or long-term, if they charge more than $50 per month, etc. This suggests that we may need more detailed information about the customer's behavior over time before maki